#03_gold_ops.quality.sql

Gold - Observabilidade / Qualidade

Este notebook consolida a saúde da camada gold para consumo em BI e operação:
- contagem de linhas por tabela gold
- competência mais recente (freshness de negócio)
- último timestamp de build (gold_build_ts)
- persiste snapshot em gold_ops.quality_reports_history

##Objetivo
- visibilidade operacional (DataOps) para a gold
- Facilita troubleshooting (tabela desatualizada, volume anormal)
- cria histórico de snapshot para tendência

##Estratégia 
- SQL-Only
- Snapshot por tabela gold (uma linha por tabela)
- não depende de TEMP VIEW para evitar problemas de sessão

##Contexto

In [0]:
USE CATALOG healthcare_dev;

##Cria schemas

In [0]:
CREATE SCHEMA IF NOT EXISTS healthcare_dev.gold;
CREATE SCHEMA IF NOT EXISTS healthcare_dev.gold_ops;

##Cria tabela quality_report_history

In [0]:
CREATE TABLE IF NOT EXISTS healthcare_dev.gold_ops.quality_report_history (
  report_ts TIMESTAMP,
  table_name STRING,

  row_count BIGINT,
  max_competencia STRING,
  max_gold_build_ts TIMESTAMP,

  status STRING,
  notes STRING
)
USING DELTA;

##Relatorio

In [0]:
CREATE OR REPLACE TEMP VIEW gold_quality_report AS
SELECT
  'mart_member_month' AS table_name,
  (SELECT COUNT(*) FROM healthcare_dev.gold.mart_member_month) AS row_count,
  (SELECT MAX(competencia) FROM healthcare_dev.gold.mart_member_month) AS max_competencia,
  (SELECT MAX(gold_build_ts) FROM healthcare_dev.gold.mart_member_month) AS max_gold_build_ts,
  'OK' AS status,
  'foundation mart (beneficiario x competencia)' AS notes
UNION ALL
SELECT
  'kpi_sinistralidade',
  (SELECT COUNT(*) FROM healthcare_dev.gold.kpi_sinistralidade),
  (SELECT MAX(competencia) FROM healthcare_dev.gold.kpi_sinistralidade),
  (SELECT MAX(gold_build_ts) FROM healthcare_dev.gold.kpi_sinistralidade),
  'OK',
  'utilizacao/custo por recortes (UF/segmento/rede/tipo/cid)'
UNION ALL
SELECT
  'kpi_churn_admin',
  (SELECT COUNT(*) FROM healthcare_dev.gold.kpi_churn_admin),
  (SELECT MAX(competencia) FROM healthcare_dev.gold.kpi_churn_admin),
  (SELECT MAX(gold_build_ts) FROM healthcare_dev.gold.kpi_churn_admin),
  'OK',
  'inadimplencia por competencia/UF/segmento'
UNION ALL
SELECT
  'kpi_experiencia',
  (SELECT COUNT(*) FROM healthcare_dev.gold.kpi_experiencia),
  (SELECT MAX(competencia) FROM healthcare_dev.gold.kpi_experiencia),
  (SELECT MAX(gold_build_ts) FROM healthcare_dev.gold.kpi_experiencia),
  'OK',
  'SAC + digital por competencia/UF/segmento'
UNION ALL
SELECT
  'mart_kpi_executivo',
  (SELECT COUNT(*) FROM healthcare_dev.gold.mart_kpi_executivo),
  (SELECT MAX(competencia) FROM healthcare_dev.gold.mart_kpi_executivo),
  (SELECT MAX(gold_build_ts) FROM healthcare_dev.gold.mart_kpi_executivo),
  'OK',
  'painel executivo (resumo mensal)'
;

##Checa sanidade

In [0]:
SELECT *
FROM gold_quality_report
ORDER BY table_name;

##Churn admin: pct entre 0 e 100

In [0]:
SELECT
  COUNT(*) AS qtd_linhas_fora_faixa
FROM healthcare_dev.gold.kpi_churn_admin
WHERE pct_inadimplencia < 0 OR pct_inadimplencia > 100;

##Sinistralidade: custo_pmpm não deve ser negativo (pode zerar)

In [0]:
SELECT
  COUNT(*) AS qtd_linhas_custo_pmpm_negativo
FROM healthcare_dev.gold.kpi_sinistralidade
WHERE custo_pmpm < 0;

##Experiencia: http_invalid_rate entre 0 e 100

In [0]:
SELECT
  COUNT(*) AS qtd_linhas_http_rate_fora_faixa
FROM healthcare_dev.gold.kpi_experiencia
WHERE http_invalid_rate < 0 OR http_invalid_rate > 100;

##Persiste snapshot no histórico

In [0]:
INSERT INTO healthcare_dev.gold_ops.quality_report_history
SELECT
  current_timestamp() AS report_ts,
  table_name,
  row_count,
  max_competencia,
  max_gold_build_ts,
  status,
  notes
FROM gold_quality_report;

##Retorna quantos snapshots foram adicionados agora

In [0]:
SELECT
  COUNT(*) AS linhas_inseridas_neste_snapshot
FROM healthcare_dev.gold_ops.quality_report_history
WHERE report_ts >= (current_timestamp() - INTERVAL 5 MINUTES);

In [0]:
SELECT
  SUM(CASE WHEN custo_estorno_total < 0 THEN 1 ELSE 0 END) AS linhas_com_estorno,
  SUM(custo_estorno_total) AS soma_estornos
FROM healthcare_dev.gold.kpi_sinistralidade;